# Predictive Modeling and Risk Scoring for Bank Customer Churn

**Project type:** Technical / Data Science Capstone
**Submitted to:** Unified Mentor
**Dataset:** European Bank customer records (10,000 customers, 3 markets — France, Germany, Spain)

---

### Objective
Build a predictive churn-intelligence system that:
- Predicts customer churn with high accuracy
- Generates calibrated churn probability (risk) scores
- Identifies the key drivers of churn with explainable, business-relevant insights

This notebook walks through the full pipeline: data preprocessing, feature engineering,
model development, evaluation, and explainability.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

## 2. Load Data & Background

Customer churn directly impacts Customer Lifetime Value (CLV), revenue stability,
cross-sell/upsell potential, and long-term competitiveness. This project moves beyond
*explaining* why churn happened, toward *predicting* who is likely to churn — enabling
proactive retention campaigns, personalized offers, and targeted engagement.

In [ ]:
df = pd.read_csv('data/European_Bank.csv')
df = df.drop(columns=['Year'])  # constant field, non-informative
print(df.shape)
df.head()

### Data quality check

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nChurn rate:", df['Exited'].mean().round(4))
df.describe().T

## 3. Exploratory Data Analysis

Key questions: Who churns? Does geography matter? Does engagement matter?
Does product holding matter?

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
counts = df['Exited'].value_counts().sort_index()
ax.bar(['Retained','Churned'], counts.values, color=['#2E5EAA','#C64B4B'])
for i,v in enumerate(counts.values):
    ax.text(i, v+50, f"{v}\n({v/len(df):.1%})", ha='center', fontweight='bold')
ax.set_title('Customer Churn Distribution', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
geo_churn = df.groupby('Geography')['Exited'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(geo_churn.index, geo_churn.values, color=['#2E5EAA','#E8743B','#5AA469'])
for i,v in enumerate(geo_churn.values):
    ax.text(i, v+0.005, f"{v:.1%}", ha='center', fontweight='bold')
ax.set_title('Churn Rate by Geography', fontweight='bold')
plt.tight_layout(); plt.show()
geo_churn

In [ ]:
prod_churn = df.groupby('NumOfProducts')['Exited'].mean()
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(prod_churn.index.astype(str), prod_churn.values, color='#5AA469')
for i,v in enumerate(prod_churn.values):
    ax.text(i, v+0.01, f"{v:.1%}", ha='center', fontweight='bold', fontsize=9)
ax.set_title('Churn Rate by Number of Products Held', fontweight='bold')
plt.tight_layout(); plt.show()
prod_churn

> **Striking finding:** customers holding 3 or 4 products churn at 83% and 100%
> respectively — versus 28% for 1 product and just 8% for 2 products. This strongly
> suggests over-selling / cross-sell mismatch (or forced bundling followed by
> dissatisfaction) rather than a simple "more products = more loyalty" relationship.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.kdeplot(df.loc[df.Exited==0,'Age'], fill=True, color='#2E5EAA', label='Retained', alpha=0.5, ax=ax)
sns.kdeplot(df.loc[df.Exited==1,'Age'], fill=True, color='#C64B4B', label='Churned', alpha=0.5, ax=ax)
ax.set_title('Age Distribution: Retained vs Churned', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
num_cols = ['CreditScore','Age','Tenure','Balance','NumOfProducts',
            'HasCrCard','IsActiveMember','EstimatedSalary','Exited']
fig, ax = plt.subplots(figsize=(8,6.5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Feature Engineering

Derived features designed to capture relationship-strength and engagement signals,
per the project brief:
- **BalanceSalaryRatio** — balance relative to income (financial capacity signal)
- **ProductDensity** — products per year of tenure (relationship depth over time)
- **EngagementProductInteraction** — active membership × number of products
- **AgeTenureInteraction** — captures life-stage / relationship-length combined effect
- **IsZeroBalance** — flags dormant/non-funded accounts

In [ ]:
df = df.drop(columns=['CustomerId','Surname'])  # identifiers, non-informative

df['BalanceSalaryRatio'] = df['Balance'] / df['EstimatedSalary'].replace(0,1)
df['ProductDensity'] = df['NumOfProducts'] / df['Tenure'].replace(0,1)
df['EngagementProductInteraction'] = df['IsActiveMember'] * df['NumOfProducts']
df['AgeTenureInteraction'] = df['Age'] * df['Tenure']
df['IsZeroBalance'] = (df['Balance'] == 0).astype(int)

TARGET = 'Exited'
categorical_features = ['Geography','Gender']
numeric_features = [c for c in df.columns if c not in categorical_features + [TARGET]]

X = df.drop(columns=[TARGET])
y = df[TARGET]
X.head()

## 5. Train-Test Split & Preprocessing

Stratified split preserves the ~20% churn class distribution in both sets, per the
project's train-test strategy.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 6. Model Development

Baseline (Logistic Regression) → tree-based models (Decision Tree, Random Forest) →
advanced ensemble (Gradient Boosting), each wrapped in a full preprocessing pipeline
and validated with 5-fold stratified cross-validation on ROC-AUC.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced',
                                             random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=250, max_depth=3, learning_rate=0.05,
                                                      random_state=RANDOM_STATE),
}

results, fitted_pipelines, roc_data = [], {}, {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:,1]
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'CV ROC-AUC (mean)': cv_auc.mean(),
    })
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, results[-1]['ROC-AUC'])

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results_df.round(4)

## 7. Model Evaluation

In [ ]:
BEST_MODEL_NAME = results_df.iloc[0]['Model']
best_pipe = fitted_pipelines[BEST_MODEL_NAME]
print("Best model:", BEST_MODEL_NAME)

fig, ax = plt.subplots(figsize=(6.5,5.5))
for name,(fpr,tpr,auc) in roc_data.items():
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=2)
ax.plot([0,1],[0,1],'--', color='gray', label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
cm = confusion_matrix(y_test, best_pipe.predict(X_test))
fig, ax = plt.subplots(figsize=(5,4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {BEST_MODEL_NAME}', fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Model Explainability

Feature importance is critical for regulatory compliance and business trust — a bank
cannot act on a "black box" score without being able to explain *why* a customer is
flagged as high-risk.

In [ ]:
perm_result = permutation_importance(best_pipe, X_test, y_test, n_repeats=15,
                                      random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1)
perm_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance_mean': perm_result.importances_mean,
}).sort_values('Importance_mean', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8,6))
top = perm_df.head(10).iloc[::-1]
ax.barh(top['Feature'], top['Importance_mean'], color='#5AA469')
ax.set_title(f'Permutation Importance — {BEST_MODEL_NAME}', fontweight='bold')
plt.tight_layout(); plt.show()
perm_df.head(10)

In [ ]:
X_train_pdp = X_train.copy()
for c in X_train_pdp.select_dtypes(include=['int64','int32']).columns:
    X_train_pdp[c] = X_train_pdp[c].astype(float)

top_features = ['Age','NumOfProducts','IsActiveMember','Balance']
fig, axes = plt.subplots(1, 4, figsize=(17,4))
PartialDependenceDisplay.from_estimator(best_pipe, X_train_pdp, top_features, ax=axes)
plt.suptitle(f'Partial Dependence — {BEST_MODEL_NAME}', fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

## 9. Key Insights & Recommendations

1. **Age is the single strongest predictor.** Churn risk rises sharply for customers
   in their late-40s to mid-60s — likely a life-stage effect (retirement planning,
   competing offers from wealth-management competitors). *Recommendation:* dedicated
   retention offers / relationship-manager outreach for the 45–65 segment.

2. **Product count is a double-edged signal.** 1 product = moderate risk, 2 products
   = lowest risk (the "sweet spot"), but 3–4 products correlates with near-certain
   churn. *Recommendation:* investigate whether 3–4 product customers were mis-sold
   bundles or are dissatisfied with fees/overlap; treat as an urgent retention flag
   rather than a loyalty signal.

3. **Geography matters — Germany churns almost 2x France/Spain.** *Recommendation:*
   local investigation — pricing, competition, or service-quality gaps in the German
   market.

4. **Inactive members churn nearly 2x more than active ones.** *Recommendation:*
   proactive engagement nudges (app usage, transaction prompts) for members who go
   quiet.

5. **Gradient Boosting was selected as the production model**, balancing strong
   ROC-AUC with practical precision — the model with the best discrimination between
   likely stayers and likely leavers.

## 10. Save Model Artifacts

In [ ]:
joblib.dump(best_pipe, 'models/best_model.pkl')
print("Saved best model:", BEST_MODEL_NAME)